#  Build a Feature Engineering Pipeline with Embeddings

In [22]:
!pip install pyspark

In [23]:
from pyspark.sql.functions import when, col
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("TelcoChurn").getOrCreate()

# 1. DOWNLOAD E PREPARAÇÃO (Ponte entre GitHub e Spark)
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df_pandas = pd.read_csv(url)
df_pandas.to_csv("telco_churn.csv", index=False)

# 2. CARREGAMENTO COM FOCO EM VALORES NULOS
# Conforme a figura, usamos a opção "nullValue" para que espaços vazios no CSV
# sejam interpretados como NULL real no Spark, permitindo o cast de 'TotalCharges'.
dataset_path = "telco_churn.csv"

telco_df = (spark.read
    .option("nullValue", " ")     # Define string com espaço como nulo (crucial para TotalCharges)
    .option("header", "true")     # Usa a primeira linha como cabeçalho
    .option("inferSchema", "true") # Tenta identificar os tipos (Double, Integer) automaticamente
    .option("multiLine", "true")  # Permite que registros ocupem mais de uma linha
    .csv(dataset_path))

# 3. SELEÇÃO DE COLUNAS DE INTERESSE
# Na figura, é feita uma filtragem para manter apenas as colunas relevantes
# para o treinamento do modelo e engenharia de recursos.
telco_df = telco_df.select(
    "gender",
    "SeniorCitizen",
    "Partner",
    "tenure",
    "InternetService",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "TotalCharges",
    "Churn"
)

# 4. VISUALIZAÇÃO
display(telco_df)

DataFrame[gender: string, SeniorCitizen: int, Partner: string, tenure: int, InternetService: string, Contract: string, PaperlessBilling: string, PaymentMethod: string, TotalCharges: double, Churn: string]

In [24]:
telco_df.show(5)

+------+-------------+-------+------+---------------+--------------+----------------+--------------------+------------+-----+
|gender|SeniorCitizen|Partner|tenure|InternetService|      Contract|PaperlessBilling|       PaymentMethod|TotalCharges|Churn|
+------+-------------+-------+------+---------------+--------------+----------------+--------------------+------------+-----+
|Female|            0|    Yes|     1|            DSL|Month-to-month|             Yes|    Electronic check|       29.85|   No|
|  Male|            0|     No|    34|            DSL|      One year|              No|        Mailed check|      1889.5|   No|
|  Male|            0|     No|     2|            DSL|Month-to-month|             Yes|        Mailed check|      108.15|  Yes|
|  Male|            0|     No|    45|            DSL|      One year|              No|Bank transfer (au...|     1840.75|   No|
|Female|            0|     No|     2|    Fiber optic|Month-to-month|             Yes|    Electronic check|      151.65

# Splitting the Dataset into Training and Testing Sets

Once the data has been cleaned, we split it into training and testing sets using an 80-20 split.
Since telco_df is a PySpark DataFrame, we will use randomSplit().

In [25]:
# Dividindo o dataframe em treino (80%) e teste (20%)
# O parâmetro seed=42 garante que a divisão seja sempre a mesma ao rodar o código novamente
train_df, test_df = telco_df.randomSplit([.8, .2], seed=42)

# Transforming the Dataset

Para garantir que todas as características numéricas e categóricas sejam compatíveis com algoritmos de machine learning, realizamos várias transformações.

Convert Integer and Boolean Columns to Double
Muitos algoritmos de machine learning requerem entrada numérica, por isso convertemos todas as colunas inteiras e booleanas para double.

Isso garante a consistência numérica no conjunto de dados.

In [26]:
from pyspark.sql.types import IntegerType, BooleanType, StringType, DoubleType
from pyspark.sql.functions import col, count, when

# 1. Obter uma lista de colunas que são do tipo Integer ou Boolean
# O script percorre o esquema (schema) do dataframe de treino para identificar esses tipos.
integer_cols = [column.name for column in train_df.schema.fields if column.dataType == IntegerType() or column.dataType == BooleanType()]

# 2. Loop através das colunas identificadas para converter cada uma para Double
# O comando .cast("double") é usado para transformar os dados.
for column in integer_cols:
    train_df = train_df.withColumn(column, col(column).cast("double"))
    test_df = test_df.withColumn(column, col(column).cast("double"))

In [27]:
# --- CÉLULA 16: IDENTIFICANDO VALORES AUSENTES EM COLUNAS NUMÉRICAS ---

from pyspark.sql.functions import count, when

# 1. Identificar colunas numéricas
# Criamos uma lista com os nomes das colunas que possuem o tipo DoubleType.
# Isso é feito percorrendo os campos do esquema (schema) do dataframe de treino.
num_cols = [c.name for c in train_df.schema.fields if c.dataType == DoubleType()]

# 2. Contar valores ausentes em colunas numéricas
# Criamos uma lógica que percorre cada coluna numérica e conta quantas vezes o valor é nulo (isNull).
# O resultado é transformado em um dicionário para facilitar a leitura.
num_missing_values_logic = [count(when(col(column).isNull(), column)).alias(column) for column in num_cols]
row_dict_num = train_df.select(num_missing_values_logic).first().asDict()

# 3. Filtrar apenas as colunas que realmente possuem valores nulos
# Criamos uma lista final contendo apenas as colunas onde a contagem de nulos é maior que zero.
num_missing_cols = [column for column in row_dict_num if row_dict_num[column] > 0]

# 4. Exibir o resultado
print(f"Numeric columns with missing values: {num_missing_cols}")

Numeric columns with missing values: ['TotalCharges']


# Creating a Feature Engineering Pipeline

A Spark ML Pipeline chains multiple transformation and estimation steps into a single, reproducible workflow. Rather than applying each step manually - which risks inconsistency between training and test data - a pipeline ensures that the exact same transformations are learned from training data and then applied uniformly to new data.

This is especially important for steps like imputation and scaling, where the statistics used for transformation (e.g., mean, standard deviation) must be derived from training data only and then applied to the test set. This prevents data leakage.

Our pipeline includes the following


Step,Transformer,Purpose

1,StringIndexer,Convert string categories to numeric indices

2,Imputer,Fill missing numerical values using the mean

3,VectorAssembler,Combine numerical columns into a single vector

4,StandardScaler,Normalize numerical feature values

5,OneHotEncoder,Convert categorical indices to binary sparse vectors

6,VectorAssembler,Combine all features into a final feature vector

# Step 1 - Encode Categorical Features: StringIndexer and OneHotEncoder


Spark ML algorithms require numeric input. To handle string (categorical) columns, we use a two-step encoding approach:

- StringIndexer converts each unique string value to a numeric index (e.g., "Male" $\rightarrow$ 0.0, "Female" $\rightarrow$ 1.0). We use handleInvalid="keep" so that null values or unseen categories are assigned a reserved index rather than raising an error.

# Explicação Técnica dos Conceitos

Este trecho do curso explica como organizar o caos das transformações de dados em uma estrutura profissional.

## 1. O que é o Pipeline?

Imagine uma linha de montagem de fábrica. O Pipeline garante que os dados brutos entrem de um lado e saiam transformados exatamente da mesma forma do outro. Se calcular a média de uma coluna manualmente no treino, mas esquecer de aplicar exatamente essa mesma média no teste, o modelo dará resultados errados. O Pipeline automatiza isso.

## 2. O perigo do Data Leakage (Vazamento de Dados):

A figura menciona que estatísticas (como a média para o Imputer) devem vir apenas do treino. Se usar a média do dataset inteiro (treino + teste), o modelo estará "espiando" o futuro (o teste), o que gera uma performance artificialmente alta que falha na vida real.

In [28]:
# --- CÉLULA 21: CODIFICAÇÃO DE VARIÁVEIS CATEGÓRICAS ---

from pyspark.ml.feature import StringIndexer, OneHotEncoder

# 1. Definição das colunas de características categóricas
# Listamos explicitamente as colunas de texto que precisam ser convertidas em números.
# Nota: A coluna alvo 'Churn' é excluída aqui para ser tratada separadamente.
categorical_cols = ["gender", "Partner", "InternetService", "Contract", "PaperlessBilling", "PaymentMethod"]

# 2. Definição dos nomes das colunas de saída
# Criamos listas para os nomes das colunas após o Indexer (_index) e após o OneHotEncoder (_ohe).
categorical_cols_indexed = [c + "_index" for c in categorical_cols]
ohe_cols = [c + "_ohe" for c in categorical_cols]

# 3. StringIndexer: converte categorias de string (texto) em índices numéricos
# Exemplo: "DSL" vira 0.0, "Fiber optic" vira 1.0.
# handleInvalid="keep": Garante que valores nulos ou novas categorias não quebrem o modelo.
string_indexer = StringIndexer(
    inputCols=categorical_cols,
    outputCols=categorical_cols_indexed,
    handleInvalid="keep"
)

# 4. OneHotEncoder: converte índices numéricos em vetores binários esparsos
# Isso evita que o modelo trate os índices (0, 1, 2) como uma ordem de importância.
# Transforma o índice 1.0 em uma coluna de 3 categorias em [0, 1, 0].
one_hot_encoder = OneHotEncoder(
    inputCols=categorical_cols_indexed,
    outputCols=ohe_cols,
    handleInvalid="keep"
)

# Step 2 - Impute Missing Numerical Values

Valores ausentes em colunas numéricas podem causar falhas nos estimadores do Spark ML. O transformador Imputer substitui os valores faltantes por uma estatística calculada — neste caso, a média (mean) de cada coluna.

Importante: O imputer é ajustado (fitted) apenas no conjunto de treinamento. Quando incluído no pipeline, os valores médios são computados do train_df durante o pipeline.fit(train_df) e depois aplicados tanto ao train_df quanto ao test_df durante o .transform(). Isso evita o vazamento de dados (data leakage) do conjunto de teste para o processo de treinamento.

In [29]:
# --- CÉLULA 23: IMPUTAÇÃO DE VALORES NUMÉRICOS ---

from pyspark.ml.feature import Imputer

# Imputar valores ausentes em colunas numéricas.
# Definimos outputCols igual a inputCols para que os valores ausentes sejam preenchidos no lugar (in-place).
imputer = Imputer(
    inputCols=num_missing_cols,  # Lista identificada na Célula 16 (ex: TotalCharges)
    outputCols=num_missing_cols,
    strategy="mean"              # Estratégia de preenchimento pela média
)

# Step 3 - Assemble and Scale Numerical Features

Antes do escalonamento, combinamos todas as colunas numéricas em um único vetor usando o VectorAssembler. O StandardScaler então padroniza os valores removendo a média e escalonando para a variância unitária.

O escalonamento garante que características com grandes intervalos numéricos (ex: TotalCharges) não dominem características com intervalos menores (ex: SeniorCitizen) em algoritmos baseados em distância ou gradiente.

In [30]:
# --- CÉLULA 25: AGRUPAMENTO E ESCALONAMENTO ---
from pyspark.ml.feature import VectorAssembler, StandardScaler

# 1. Combinar todas as colunas numéricas em um único vetor
# 'num_cols' é a lista de colunas numéricas definida anteriormente.
numerical_assembler = VectorAssembler(
    inputCols=num_cols,
    outputCol="numerical_assembled"
)

# 2. Escalonar as características numéricas para padronizar os valores
numerical_scaler = StandardScaler(
    inputCol="numerical_assembled",
    outputCol="numerical_scaled"
)

# Step 4 - Assemble the Final Feature Vector

O VectorAssembler final combina as características numéricas escalonadas e os vetores categóricos (one-hot encoded) em um único vetor de características chamado all_features. Os modelos de Spark ML esperam que todas as características de entrada estejam compactadas em uma única coluna de vetor neste formato.


In [31]:
# --- CÉLULA 27: MONTAGEM DO VETOR FINAL ---

# 1. Definir a lista de colunas que irão compor o vetor final
# Combinamos a coluna numérica já escalonada com as colunas categóricas processadas (ohe_cols)
feature_cols = ["numerical_scaled"] + ohe_cols

# 2. Configurar o VectorAssembler para criar a coluna 'all_features'
vector_assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="all_features"
)

# Step 5 - Build the Pipeline

Nesta etapa, combinamos todos os passos de transformação em um único Pipeline. A ordem dos estágios é fundamental — por exemplo, o StringIndexer deve rodar antes do OneHotEncoder, e o VectorAssembler numérico deve rodar antes do StandardScaler.

Uma vez definido, o pipeline pode ser ajustado (fitted) uma única vez nos dados de treinamento e aplicado muitas vezes a novos dados, tornando-o confiável e pronto para produção.

In [32]:
# --- CÉLULA 28: INSTANCIAÇÃO DO PIPELINE ---

from pyspark.ml import Pipeline

# 1. Definir a sequência ordenada dos estágios do pipeline
# A ordem aqui é crítica pois cada estágio depende do output do anterior.
stages_list = [
    string_indexer,      # Converte strings em índices
    imputer,             # Preenche valores nulos
    numerical_assembler, # Agrupa colunas numéricas
    numerical_scaler,    # Escala as colunas numéricas
    one_hot_encoder,     # Codifica índices em vetores binários
    vector_assembler     # Junta tudo no vetor final 'all_features'
]

# 2. Instanciar o pipeline com a lista de estágios definida
pipeline = Pipeline(stages=stages_list)

# Fit the Pipeline

Ajustar (fitting) o pipeline nos dados de treinamento executa cada estágio do tipo Estimator para aprender seus parâmetros (ex: valores médios para o Imputer, fatores de escala para o StandardScaler, mapeamentos de índices para o StringIndexer). Esses parâmetros aprendidos são armazenados no PipelineModel resultante e reutilizados consistentemente ao transformar novos dados.

Transformers (ex: VectorAssembler, OneHotEncoder) não aprendem parâmetros dos dados — eles são aplicados diretamente.

# O que acontece durante o Fitting?

Quando chamamos .fit(train_df), cada estágio no pipeline processa os dados em sequência:

StringIndexer: Escaneia os dados de treinamento para construir um mapeamento de índice de cada categoria de string única para um valor numérico. Esses mapeamentos são salvos e usados consistentemente em novos dados.

Imputer: Computa a média para cada coluna numérica especificada de train_df e armazena esses valores para preencher dados ausentes durante a transformação.

VectorAssembler (numérico): Combina as colunas numéricas especificadas em um único vetor — nenhum aprendizado é necessário.

StandardScaler: Computa a média e o desvio padrão do vetor numérico montado a partir dos dados de treinamento, usados para normalizar os valores das características.

OneHotEncoder: Converte índices de categorias numéricas em vetores binários esparsos — nenhum aprendizado é necessário.

VectorAssembler (final): Une todos os vetores de características na coluna única all_features — nenhum aprendizado é necessário.

In [33]:
# --- CÉLULA 32: TREINAMENTO DO PIPELINE ---

# Ajustar o pipeline nos dados de treinamento - aprende todas as estatísticas necessárias
# O resultado é um PipelineModel (objeto treinado)
pipeline_model = pipeline.fit(train_df)

# Apply the Feature Engineering Pipeline

Uma vez que o pipeline está ajustado (fitted) aos dados de treinamento, ele pode ser aplicado a qualquer dataset usando o método .transform(). Aplicamos o pipeline em ambos os conjuntos:

Train Dataset (train_df): Gera as características de treinamento transformadas.

Test Dataset (test_df): Garante que as mesmas transformações sejam aplicadas de forma consistente.

O resultado é um dataset transformado contendo o vetor final all_features, pronto para a modelagem.

In [34]:
# --- CÉLULA 34: APLICANDO O PIPELINE TREINADO ---

# Transformar os datasets de treino e teste usando o pipeline ajustado (pipeline_model)
train_transformed_df = pipeline_model.transform(train_df)
test_transformed_df = pipeline_model.transform(test_df)

Script de Visualização dos Vetores (Célula 35):

In [35]:
# --- CÉLULA 35: VISUALIZAÇÃO DOS RESULTADOS ---

# Mostrar uma amostra dos vetores de características transformados
# O parâmetro truncate=False permite ver o vetor completo na coluna 'all_features'
train_transformed_df.select("all_features").show(3, truncate=False)

+---------------------------------------------------------------------------------------------+
|all_features                                                                                 |
+---------------------------------------------------------------------------------------------+
|(25,[1,2,4,6,10,13,18,23],[0.04063920218920153,0.011121121600887504,1.0,1.0,1.0,1.0,1.0,1.0])|
|(25,[1,2,4,6,10,13,18,23],[0.04063920218920153,0.013191191760260623,1.0,1.0,1.0,1.0,1.0,1.0])|
|(25,[1,2,4,6,10,13,18,23],[0.04063920218920153,0.021823824765305973,1.0,1.0,1.0,1.0,1.0,1.0])|
+---------------------------------------------------------------------------------------------+
only showing top 3 rows


# Prepare the Target Column

Os modelos de Spark ML exigem que a coluna alvo (label) seja numérica. Em nosso conjunto de dados, a coluna Churn contém valores de string — "Yes" ou "No". Nós os convertemos para uma representação numérica antes de passar os dados para um modelo.

Utilizamos o seguinte mapeamento:

"Yes" $\rightarrow$ 1.0 (cliente cancelou/churned)

"No" $\rightarrow$ 0.0 (cliente não cancelou)

Esta etapa é aplicada após a transformação do pipeline porque o Churn foi intencionalmente excluído do pipeline de engenharia de recursos — ele é o nosso alvo de previsão, não uma característica de entrada (feature).

In [36]:
# --- CÉLULA 37: CONVERSÃO DA VARIÁVEL ALVO ---

from pyspark.sql.functions import when, col

# Converter a label de string Churn para numérica (0.0 = Sem churn, 1.0 = Churned)
# Criamos uma nova coluna chamada "label" baseada na condição da coluna "Churn"
train_prepared_df = train_transformed_df.withColumn("label", when(col("Churn") == "Yes", 1.0).otherwise(0.0))
test_prepared_df = test_transformed_df.withColumn("label", when(col("Churn") == "Yes", 1.0).otherwise(0.0))

# Exibir o dataset de treinamento final preparado
# Selecionamos apenas o vetor 'all_features' (entradas) e a 'label' (saída)
display(train_prepared_df.select("all_features", "label"))

DataFrame[all_features: vector, label: double]

# Save and Reuse the Pipeline

Preservar o pipeline de engenharia de recursos — incluindo todos os parâmetros aprendidos e a lógica de transformação — é essencial para manter a reprodutibilidade, permitir o controle de versão e facilitar a colaboração. Nesta seção, irá:

Salvar o Pipeline: Salvar o modelo de pipeline ajustado no diretório de trabalho designado.

Explorar Estágios do Pipeline Carregado: Inspecionar os estágios para confirmar a sequência de transformações aplicadas.

In [37]:
# Salvar o modelo de pipeline com modo de sobrescrita
# Isso salva o pipeline_model (que contém as médias e índices aprendidos) no caminho especificado
#pipeline_model.write().overwrite().save(f"{DA.paths.working_dir}/spark_pipelines")
#print(f"Saved model to: {DA.paths.working_dir}/spark_pipelines")

pipeline_model.write().overwrite().save(f"/spark_pipelines")
print(f"Saved model to: /spark_pipelines")

Saved model to: /spark_pipelines


# Load and Use Saved Model

Uma das maiores vantagens do Spark ML é a capacidade de carregar um pipeline salvo em uma sessão diferente ou em um ambiente de produção para transformar novos dados de forma idêntica.

In [38]:
# Carregar o modelo de pipeline salvo
from pyspark.ml import PipelineModel

# Carrega o modelo do diretório onde foi salvo anteriormente
#loaded_pipeline = PipelineModel.load(f"{DA.paths.working_dir}/spark_pipelines")
loaded_pipeline = PipelineModel.load(f"/spark_pipelines")

# Mostrar os estágios do pipeline
# Isso permite verificar se todos os transformadores (Imputer, Scaler, etc.) estão presentes
loaded_pipeline.stages

[StringIndexerModel: uid=StringIndexer_b71a2f572bea, handleInvalid=keep, numInputCols=6, numOutputCols=6,
 ImputerModel: uid=Imputer_d1858e9aba72, strategy=mean, missingValue=NaN, numInputCols=1, numOutputCols=1,
 VectorAssembler_48346d36b489,
 StandardScalerModel: uid=StandardScaler_0dcde6c626d0, numFeatures=3, withMean=false, withStd=true,
 OneHotEncoderModel: uid=OneHotEncoder_9678e62546c5, dropLast=true, handleInvalid=keep, numInputCols=6, numOutputCols=6,
 VectorAssembler_1294df3714ab]

# Using Saved Pipeline for Reuse

Embora já tenhamos aplicado o pipeline anteriormente nesta demonstração, nós recarregamos o pipeline salvo e o aplicamos novamente aqui para ilustrar como os pipelines salvos podem ser reutilizados em produção.

In [39]:
# --- CÉLULA 44: REUTILIZANDO O PIPELINE SALVO ---

# Usar o pipeline carregado para transformar o dataset de teste
# O método .transform() aplica todas as regras aprendidas (médias, escalas, índices)
test_transformed_df = loaded_pipeline.transform(test_df)

# Exibir o dataframe resultante com as transformações aplicadas
display(test_transformed_df)

DataFrame[gender: string, SeniorCitizen: double, Partner: string, tenure: double, InternetService: string, Contract: string, PaperlessBilling: string, PaymentMethod: string, TotalCharges: double, Churn: string, gender_index: double, Partner_index: double, InternetService_index: double, Contract_index: double, PaperlessBilling_index: double, PaymentMethod_index: double, numerical_assembled: vector, numerical_scaled: vector, gender_ohe: vector, Partner_ohe: vector, InternetService_ohe: vector, Contract_ohe: vector, PaperlessBilling_ohe: vector, PaymentMethod_ohe: vector, all_features: vector]

In [40]:
test_transformed_df.show(10)

+------+-------------+-------+------+---------------+--------------+----------------+--------------------+------------+-----+------------+-------------+---------------------+--------------+----------------------+-------------------+-------------------+--------------------+-------------+-------------+-------------------+-------------+--------------------+-----------------+--------------------+
|gender|SeniorCitizen|Partner|tenure|InternetService|      Contract|PaperlessBilling|       PaymentMethod|TotalCharges|Churn|gender_index|Partner_index|InternetService_index|Contract_index|PaperlessBilling_index|PaymentMethod_index|numerical_assembled|    numerical_scaled|   gender_ohe|  Partner_ohe|InternetService_ohe| Contract_ohe|PaperlessBilling_ohe|PaymentMethod_ohe|        all_features|
+------+-------------+-------+------+---------------+--------------+----------------+--------------------+------------+-----+------------+-------------+---------------------+--------------+-------------------

Reuso em Produção: O código mostra que, uma vez que você salvou o loaded_pipeline, pode aplicá-lo a qualquer novo conjunto de dados (neste caso, o test_df) sem precisar redefinir ou treinar os estágios novamente.

Inspeção dos Estágios: O output da célula 42 na imagem detalha exatamente o que foi carregado:

StringIndexerModel: Mapeamento de categorias de texto.

ImputerModel: Lógica para preenchimento de valores nulos (estratégia de média).

VectorAssembler: Agrupamento de colunas.

StandardScalerModel: Fatores de normalização (escala).

OneHotEncoderModel: Codificação binária de categorias.

# O Passo a Passo Resumido

StringIndexer: Transforma textos (categorias como "Masculino"/"Feminino") em números (índices como 0, 1, 2). É o primeiro passo porque o computador só entende cálculos matemáticos.

Imputer: Identifica buracos nos dados (valores nulos) e os preenche com a média daquela coluna específica. Isso evita que o modelo dê erro ao encontrar dados vazios.

VectorAssembler (Numérico): Pega todas as colunas que já eram números (ou que o Imputer limpou) e as junta em uma única "lista" ou vetor.

StandardScaler: Ajusta a escala dos números para que fiquem equilibrados (ex: evita que uma coluna com valores de 0 a 10.000 "atropele" uma de 0 a 1).

OneHotEncoder: Transforma os índices criados no passo 1 em vetores binários (0 e 1). Isso impede que o modelo ache que a categoria "2" é maior ou mais importante que a "0".

VectorAssembler (Final): É o "empacotador" final. Ele pega os números escalonados (passo 4) e as categorias binárias (passo 5) e junta tudo em um único vetor chamado all_features, que é o que o modelo de IA realmente vai ler.